# Sector Rotation System — Setup & Backtest

This notebook runs on **Google Colab** (free tier). It:

1. Clones the repo and installs dependencies
2. Populates the database with 2 years of historical data
3. Runs the full pipeline (regime → optimizer → screener → NLP)
4. Executes a walk-forward backtest and displays results
5. Shows the current Executive Summary with dollar allocations

**Runtime:** ~5–8 minutes on Colab free tier.

## 1. Clone Repo & Install Dependencies

In [ ]:
# Clone the repository (replace with your fork URL)
!git clone https://github.com/YOUR_USERNAME/sector-rotation-system.git
%cd sector-rotation-system

# Install dependencies
!pip install -q -r requirements.txt

## 2. Set API Keys (Optional)

FRED key is optional — the system works without it using placeholder macro data.
Set it here if you have one.

In [ ]:
import os

# Optional: set your FRED API key (free at https://fred.stlouisfed.org)
# os.environ['FRED_API_KEY'] = 'your_key_here'

# Verify
fred_key = os.environ.get('FRED_API_KEY', '')
print(f"FRED_API_KEY: {'set (' + fred_key[:4] + '...)' if fred_key else 'NOT SET (will use dummy macro data)'}")

## 3. Backfill Historical Data

Downloads ~2 years of daily prices for all ETFs and watchlist stocks via yfinance,
macro data from FRED (if key is set), and recent SEC filings.

In [ ]:
import yaml
import sqlite3
import pandas as pd

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Config loaded. Portfolio total: ${:,.0f}".format(config['portfolio']['total_value']))
print(f"  Taxable: ${config['portfolio']['accounts']['taxable']['value']:,.0f}")
print(f"  Roth IRA: ${config['portfolio']['accounts']['roth_ira']['value']:,.0f}")
print(f"  Sector ETFs: {len(config['tickers']['sector_etfs'])}")
print(f"  Watchlist tickers: {sum(len(config['tickers'].get(k, [])) for k in ['watchlist_biotech', 'watchlist_ai_software', 'watchlist_defense', 'watchlist_green_materials'])}")

In [ ]:
# Run data backfill
from data_feeds import backfill_prices, fetch_macro_data, fetch_sec_filings

print("=== Backfilling price data (this takes 1-2 min) ===")
price_result = backfill_prices(config)
print(f"Prices loaded: {price_result.get('rows_inserted', 'N/A')} rows")

print("\n=== Fetching macro data ===")
macro_result = fetch_macro_data(config)
print(f"Macro series loaded: {macro_result.get('series_count', 'N/A')}")

print("\n=== Fetching SEC filings ===")
filing_result = fetch_sec_filings(config)
print(f"Filings loaded: {filing_result.get('filings_count', 'N/A')}")

# Show DB size
conn = sqlite3.connect('rotation_system.db')
for table in ['prices', 'macro_data', 'filings']:
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f"  {table}: {count:,} rows")
conn.close()

## 4. Detect Current Regime

In [ ]:
from regime_detector import detect_regime

regime_result = detect_regime(config)

print("=" * 60)
print("CURRENT REGIME")
print("=" * 60)
print(f"  Regime:              {regime_result.get('regime', 'N/A')}")
print(f"  Wedge Volume:        {regime_result.get('wedge_volume', 'N/A'):.4f}")
print(f"  Percentile:          {regime_result.get('percentile', 'N/A'):.1f}")
print(f"  Fast Shock Active:   {regime_result.get('fast_shock', False)}")
print(f"  Confidence:          {regime_result.get('confidence', 'N/A')}")
print(f"  Consecutive Days:    {regime_result.get('consecutive_days', 'N/A')}")

## 5. Run CVaR Optimization

In [ ]:
from portfolio_optimizer import optimize_portfolio

opt_result = optimize_portfolio(config, regime_result['regime'])

total = config['portfolio']['total_value']
taxable = config['portfolio']['accounts']['taxable']['value']
roth = config['portfolio']['accounts']['roth_ira']['value']

print("=" * 60)
print(f"CVaR-OPTIMIZED ALLOCATION (Regime: {regime_result['regime']})")
print("=" * 60)
print(f"{'Asset Class':<25} {'Weight':>8} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}")
print("-" * 65)

allocations = opt_result.get('allocations', {})
for asset, weight in sorted(allocations.items(), key=lambda x: -x[1]):
    total_dollars = weight * total
    # Simplified split — actual system uses tax-location rules
    tax_dollars = weight * taxable
    roth_dollars = weight * roth
    print(f"{asset:<25} {weight:>7.1%} {total_dollars:>10,.0f} {tax_dollars:>10,.0f} {roth_dollars:>10,.0f}")

## 6. Walk-Forward Backtest

Simulates the system over the historical period, rebalancing monthly
based on the regime detected at each rebalance date.

In [ ]:
import numpy as np
import datetime as dt

conn = sqlite3.connect('rotation_system.db')

# Load SPY as benchmark
spy = pd.read_sql(
    "SELECT date, close FROM prices WHERE ticker='SPY' ORDER BY date",
    conn, parse_dates=['date']
).set_index('date')
spy_returns = spy['close'].pct_change().dropna()

# Load signals for regime history
signals = pd.read_sql(
    "SELECT date, signal_data FROM signals WHERE signal_type='regime' ORDER BY date",
    conn, parse_dates=['date']
)
conn.close()

if len(signals) > 0 and len(spy_returns) > 0:
    # Parse regime from signal_data
    import json
    signals['regime'] = signals['signal_data'].apply(
        lambda x: json.loads(x).get('regime', 'offense') if isinstance(x, str) else 'offense'
    )
    
    # Simplified backtest: map regime to equity exposure
    regime_exposure = {'offense': 0.75, 'defense': 0.35, 'panic': 0.05}
    
    signals = signals.set_index('date')
    merged = spy_returns.to_frame('spy_ret').join(signals['regime'], how='left')
    merged['regime'] = merged['regime'].ffill().fillna('offense')
    merged['exposure'] = merged['regime'].map(regime_exposure)
    merged['strategy_ret'] = merged['spy_ret'] * merged['exposure']
    
    # Cumulative returns
    merged['spy_cum'] = (1 + merged['spy_ret']).cumprod()
    merged['strategy_cum'] = (1 + merged['strategy_ret']).cumprod()
    
    # Metrics
    days = len(merged)
    ann_factor = 252 / days if days > 0 else 1
    
    spy_total = merged['spy_cum'].iloc[-1] - 1
    strat_total = merged['strategy_cum'].iloc[-1] - 1
    
    spy_ann = (1 + spy_total) ** ann_factor - 1
    strat_ann = (1 + strat_total) ** ann_factor - 1
    
    spy_vol = merged['spy_ret'].std() * np.sqrt(252)
    strat_vol = merged['strategy_ret'].std() * np.sqrt(252)
    
    spy_sharpe = spy_ann / spy_vol if spy_vol > 0 else 0
    strat_sharpe = strat_ann / strat_vol if strat_vol > 0 else 0
    
    # Max drawdown
    def max_dd(cum_series):
        peak = cum_series.cummax()
        dd = (cum_series - peak) / peak
        return dd.min()
    
    spy_mdd = max_dd(merged['spy_cum'])
    strat_mdd = max_dd(merged['strategy_cum'])
    
    mclean_decay = config['factor_model']['mclean_pontiff_decay']
    alpha_raw = strat_ann - spy_ann
    alpha_adj = alpha_raw * mclean_decay
    
    mp_label = config['backtest']['mclean_pontiff_label']
    
    print("=" * 70)
    print("WALK-FORWARD BACKTEST RESULTS")
    print("=" * 70)
    print(f"Period: {merged.index[0].strftime('%Y-%m-%d')} to {merged.index[-1].strftime('%Y-%m-%d')} ({days} trading days)")
    print()
    print(f"{'Metric':<30} {'SPY (B&H)':>15} {'Strategy':>15}")
    print("-" * 60)
    print(f"{'Total Return':<30} {spy_total:>14.1%} {strat_total:>14.1%}")
    print(f"{'Annualized Return':<30} {spy_ann:>14.1%} {strat_ann:>14.1%}")
    print(f"{'Annualized Volatility':<30} {spy_vol:>14.1%} {strat_vol:>14.1%}")
    print(f"{'Sharpe Ratio':<30} {spy_sharpe:>14.2f} {strat_sharpe:>14.2f}")
    print(f"{'Max Drawdown':<30} {spy_mdd:>14.1%} {strat_mdd:>14.1%}")
    print()
    print(f"Raw Alpha vs SPY:      {alpha_raw:>+.2%}")
    print(f"Adjusted Alpha (×{mclean_decay}): {alpha_adj:>+.2%}")
    print(f"  {mp_label}")
else:
    print("Not enough data for backtest. Run data_feeds.py --backfill first.")

## 7. Stock Screener — Top Picks

In [ ]:
from stock_screener import run_screener

screener_result = run_screener(config, regime_result['regime'])

print("=" * 60)
print("THEMATIC WATCHLIST SCORES")
print("=" * 60)

for watchlist_name in ['biotech', 'ai_software', 'defense', 'green_materials']:
    picks = screener_result.get(watchlist_name, [])
    if picks:
        print(f"\n--- {watchlist_name.upper()} ---")
        print(f"{'Ticker':<8} {'Score':>8} {'Signal':>12} {'Valuation':>15}")
        for p in sorted(picks, key=lambda x: -x.get('composite_score', 0))[:5]:
            print(f"{p.get('ticker', '?'):<8} {p.get('composite_score', 0):>7.2f} "
                  f"{p.get('signal', 'N/A'):>12} {p.get('valuation_label', 'N/A'):>15}")

## 8. Run Full Monitor (Executive Summary)

In [ ]:
# Run the full monitor pipeline and display the executive summary
!python monitor.py --run-daily 2>&1 | head -100

## 9. Download Database

Download the populated database to your local machine for use with
the Streamlit dashboard or further analysis.

In [ ]:
from google.colab import files

# Download the populated database
files.download('rotation_system.db')
print("Database downloaded. Place it in the repo root for the dashboard.")